# 总光时改正（SR + GR）：T_total = T_SR + T_PM + T_HM + T_SM

根据 Yan et al. (2021)，一程 light-time correction 可分解为：
T = T_SR + T_GR，其中 T_GR = T_PM + T_HM + T_SM（并可忽略 Δt_media）。citeturn0search0

本 Notebook 读取同一文件夹内的 4 个 CSV：
- `Shapiro_TPM_GNI1B_gamma_dtSR.csv`（T_PM）
- `THM_EIGEN6C4_L120（1）.csv`（T_HM）
- `TSM_GNI1B.csv`（T_SM）
- `T_SR_Eq34.csv`（T_SR）

按 `gps_time` 对齐后计算：
- `T_GR_s = T_PM_s + T_HM_s + T_SM_s`
- `T_total_s = T_SR_s + T_GR_s`
并给出距离域：
- `rho_GR_m = c0 * T_GR_s`
- `rho_total_m = c0 * T_total_s`

输出：
- `LightTime_Total_PM_HM_SM_SR.xlsx`


In [7]:

import pandas as pd

C0 = 299792458.0

TPM_CSV = "LightTime_T_GR_TpMr.xlsx"
TSR_CSV = "T_SR_Eq34_D_emit_C_recv_d0.xlsx"

OUT_XLSX = "T_TpMr.xlsx"

tpm = pd.read_excel(TPM_CSV)

tsr = pd.read_excel(TSR_CSV)

print("TPM:", tpm.shape, list(tpm.columns))

print("TSR:", tsr.shape, list(tsr.columns))


TPM: (86400, 23) ['gps_time', 'T_PM_s', 'rho_PM_m', 'dt_inst_s', 'dt_corr_s', 'd0_x', 'd0_y', 'd0_z', 'd0_dot_vC_mps', 'aC_x_mps2', 'aC_y_mps2', 'aC_z_mps2', 'aC_mag_mps2', 'r_e_x_m', 'r_e_y_m', 'r_e_z_m', 'T_HM_s', 'rho_HM_m', 'T_GR_s', 'rho_total_m', 'rho_sum_m', 'rho_diff_m', 'delta_t_s']
TSR: (86400, 20) ['gps_time', 'T_SR_s', 'rho_SR_m', 'dt_inst_s', 'dt_corr_s', 'T_GR_s', 're_x', 're_y', 're_z', 'd0_x', 'd0_y', 'd0_z', 'd0_dot_vC_mps', 'd0_dot_aC_mps2', 'vC2_m2ps2', 'vC_dot_aC_m2ps3', 'aC_x_mps2', 'aC_y_mps2', 'aC_z_mps2', 'aC_mag_mps2']


In [8]:

# 只取需要的列，避免重复/无关列
tpm_use = tpm[["gps_time","delta_t_s"]].copy()

tsr_use = tsr[["gps_time","T_SR_s"]].copy()

merged = (
    tpm_use .merge(tsr_use, on="gps_time", how="inner")
)

# GR 部分（PM+HM+SM）
merged["T_GR_s"] = merged["delta_t_s"]
merged["rho_GR_m"] = C0 * merged["T_GR_s"]

# 总光时改正（SR+GR）
merged["T_total_s"] = merged["T_SR_s"] + merged["T_GR_s"]
merged["rho_total_m"] = C0 * merged["T_total_s"]

# 自检：距离域也可分别求和（应与 c0*T 一致）
# merged["rho_GR_sum_m"] = merged["rho_PM_m"]
# merged["rho_total_sum_m"] = merged["rho_GR_sum_m"] + merged["rho_SR_m"]
# merged["rho_GR_diff_m"] = merged["rho_GR_m"] - merged["rho_GR_sum_m"]
# merged["rho_total_diff_m"] = merged["rho_total_m"] - merged["rho_total_sum_m"]

merged.head()


,gps_time,delta_t_s,T_SR_s,T_GR_s,rho_GR_m,T_total_s,rho_total_m
0,707659200,8.420871e-13,-1.656052e-08,8.420871e-13,0.000252,-1.655968e-08,-4.964468
1,707659201,8.420898e-13,-1.656052e-08,8.420898e-13,0.000252,-1.655968e-08,-4.964467
2,707659202,8.420925e-13,-1.656052e-08,8.420925e-13,0.000252,-1.655968e-08,-4.964466
3,707659203,8.420952e-13,-1.656052e-08,8.420952e-13,0.000252,-1.655967e-08,-4.964465
4,707659204,8.420979e-13,-1.656051e-08,8.420979e-13,0.000252,-1.655967e-08,-4.964465


In [9]:

merged.to_excel(OUT_XLSX, index=False)

print("Wrote:", OUT_XLSX)
print("rows:", len(merged))
# print("rho_GR_diff_m abs max:", merged["rho_GR_diff_m"].abs().max())
# print("rho_total_diff_m abs max:", merged["rho_total_diff_m"].abs().max())

# quick look
merged[["gps_time","T_SR_s","T_GR_s","T_total_s","rho_total_m"]].head(10)


Wrote: T_TpMr.xlsx
rows: 86400


,gps_time,T_SR_s,T_GR_s,T_total_s,rho_total_m
0,707659200,-1.656052e-08,8.420871e-13,-1.655968e-08,-4.964468
1,707659201,-1.656052e-08,8.420898e-13,-1.655968e-08,-4.964467
2,707659202,-1.656052e-08,8.420925e-13,-1.655968e-08,-4.964466
3,707659203,-1.656052e-08,8.420952e-13,-1.655967e-08,-4.964465
4,707659204,-1.656051e-08,8.420979e-13,-1.655967e-08,-4.964465
5,707659205,-1.656051e-08,8.421005e-13,-1.655967e-08,-4.964464
6,707659206,-1.656051e-08,8.421032e-13,-1.655967e-08,-4.964463
7,707659207,-1.656050e-08,8.421058e-13,-1.655966e-08,-4.964462
8,707659208,-1.656050e-08,8.421085e-13,-1.655966e-08,-4.964461
9,707659209,-1.656050e-08,8.421111e-13,-1.655966e-08,-4.964460
